# Construct Dow returns

In [1]:
import numpy as np
import pandas as pd

## Components

In [2]:
# Read and format Dow components

comps = pd.read_excel('Dow components.xlsx')

comps = comps.set_index('Unnamed: 0')
comps.index.name = 'company'

comps = comps.drop('National Cash Register')

idx = [str(c)[:10] for c in comps.columns]
comps = comps.T
comps.index = pd.to_datetime(pd.Series(idx))

comps = comps.unstack().dropna().reset_index()
comps = comps.rename(columns={'level_1':'date', 0:'permno'})
comps['permno'] = pd.to_numeric(comps['permno'], downcast='integer')

In [3]:
comps

,company,date,permno
0,Primerica,1988-12-16,70519
1,Swift & Company,1959-06-01,19713
2,United Air Transport,1930-07-18,17830
3,Victor Talking Machine,1928-10-01,16002
4,Remington Typewriter,1925-12-31,14525
...,...,...,...
1320,General Electric,2008-09-22,12060
1321,General Electric,2009-06-08,12060
1322,General Electric,2012-09-24,12060
1323,General Electric,2013-09-23,12060


In [4]:
comps.groupby('date')['permno'].count()

date
1925-12-31    20
1927-03-16    20
1928-10-01    30
1929-01-08    28
1929-09-14    29
1930-01-29    29
1930-07-18    29
1932-05-26    30
1933-08-15    30
1934-08-13    30
1935-11-20    30
1937-01-08    30
1939-03-14    30
1956-07-03    30
1959-04-22    30
1959-06-01    30
1976-04-21    30
1976-08-09    30
1979-06-29    30
1982-08-30    30
1985-09-19    30
1985-10-30    30
1986-07-08    30
1987-03-12    30
1988-12-16    30
1991-05-06    30
1994-04-20    30
1997-03-17    30
1999-01-04    30
1999-11-01    30
2000-12-30    30
2002-04-08    30
2003-01-27    30
2004-04-08    30
2005-11-21    30
2008-02-19    30
2008-09-22    30
2009-06-08    30
2012-09-24    30
2013-09-23    30
2015-03-06    30
2018-06-26    30
2019-04-02    30
2020-04-06    30
2020-08-31    30
Name: permno, dtype: int64

In [5]:
# For each year, we want the date of the most recent update to the list

dates = pd.Series(comps['date'].unique())

# keep last date each year
dates = dates.groupby(dates.dt.year).max()

dates = dates.reindex(index=np.arange(1925,2022))

dates = dates.fillna(method='ffill')

dates

1925   1925-12-31
1926   1925-12-31
1927   1927-03-16
1928   1928-10-01
1929   1929-09-14
          ...    
2017   2015-03-06
2018   2018-06-26
2019   2019-04-02
2020   2020-08-31
2021   2020-08-31
Length: 97, dtype: datetime64[ns]

## WRDS

In [6]:
import wrds

conn = wrds.Connection()

In [6]:
# restrict queries to permnos that are in the Dow at some point

params = {'permnos': (tuple([str(permno) for permno in np.unique(comps['permno'])]))}

Loading library list...
Done


### Returns

In [8]:
rets = conn.raw_sql("""
    SELECT permno, mthcaldt, mthprc, mthprevprc, mthret, mthretx
    FROM crsp.msf_v2
    WHERE permno IN %(permnos)s
    """,
    date_cols=['mthcaldt'], params=params)

rets['permno'] = pd.to_numeric(rets['permno'], downcast='integer')

rets = rets.sort_values(['permno', 'mthcaldt'])

In [9]:
rets

,permno,mthcaldt,mthprc,mthprevprc,mthret,mthretx
0,10006,1925-12-31,109.000,NaN,NaN,NaN
1,10006,1926-01-30,110.250,109.000,0.032732,0.032732
2,10006,1926-02-27,102.375,110.250,-0.071429,-0.071429
3,10006,1926-03-31,96.500,102.375,-0.043212,-0.057387
4,10006,1926-04-30,94.000,96.500,-0.025907,-0.025907
...,...,...,...,...,...,...
71070,92655,2022-02-28,475.870,472.570,0.006983,0.006983
71071,92655,2022-03-31,509.970,475.870,0.074876,0.071658
71072,92655,2022-04-29,508.550,509.970,-0.002784,-0.002784
71073,92655,2022-05-31,496.780,508.550,-0.023144,-0.023144


In [10]:
annrets = rets.groupby(['permno', rets['mthcaldt'].dt.year])[['mthret', 'mthretx']].apply(lambda x: (1+x).product()-1)

prcs = rets.groupby(['permno', rets['mthcaldt'].dt.year])[['mthprc']].last()

annrets = annrets.join(prcs).reset_index()

annrets = annrets.rename(columns={'mthcaldt':'year', 'mthret':'ret', 'mthretx':'retx', 'mthprc':'prc'})

In [11]:
annrets

,permno,year,ret,retx,prc
0,10006,1925,0.000000,0.000000,109.000
1,10006,1926,0.008753,-0.049231,101.500
2,10006,1927,0.150657,0.086207,110.250
3,10006,1928,-0.054262,-0.109977,98.125
4,10006,1929,-0.153677,-0.205096,78.000
...,...,...,...,...,...
6031,92655,2018,0.145207,0.130001,249.120
6032,92655,2019,0.199871,0.180074,293.980
6033,92655,2020,0.212048,0.192870,350.680
6034,92655,2021,0.452131,0.431904,502.140


### Dividends

This appears to be more accurate than using `ret - retx`.

In [12]:
sql = """
    SELECT permno, date_part('year', exdt) as year, distcd, sum(divamt) as div
    FROM crsp.msedist
    WHERE permno IN %(permnos)s
        AND divamt>0
    GROUP BY permno, year, distcd
    ORDER BY permno, year, distcd
    """

divs = conn.raw_sql(sql, params=params)
divs = divs.dropna()

divs[['permno', 'year', 'distcd']] = divs[['permno', 'year', 'distcd']].apply(lambda x: pd.to_numeric(x, downcast='integer'))

divs['distcd'] = divs['distcd'].astype(str)

In [13]:
# aggregate all dividend types

divs = divs.groupby(['permno', 'year'])['div'].sum().reset_index()

In [14]:
divs.head()

,permno,year,div
0,10006,1926,8.3125
1,10006,1927,6.0000
2,10006,1928,6.0000
3,10006,1929,6.0000
4,10006,1930,6.0000


In [15]:
annrets.head()

,permno,year,ret,retx,prc
0,10006,1925,0.000000,0.000000,109.000
1,10006,1926,0.008753,-0.049231,101.500
2,10006,1927,0.150657,0.086207,110.250
3,10006,1928,-0.054262,-0.109977,98.125
4,10006,1929,-0.153677,-0.205096,78.000


In [16]:
annretsdivs = pd.merge(annrets, divs, how='left')
annretsdivs['div'] = annretsdivs['div'].fillna(0)

In [17]:
annretsdivs

,permno,year,ret,retx,prc,div
0,10006,1925,0.000000,0.000000,109.000,0.0000
1,10006,1926,0.008753,-0.049231,101.500,8.3125
2,10006,1927,0.150657,0.086207,110.250,6.0000
3,10006,1928,-0.054262,-0.109977,98.125,6.0000
4,10006,1929,-0.153677,-0.205096,78.000,6.0000
...,...,...,...,...,...,...
6031,92655,2018,0.145207,0.130001,249.120,3.4500
6032,92655,2019,0.199871,0.180074,293.980,4.1400
6033,92655,2020,0.212048,0.192870,350.680,4.8300
6034,92655,2021,0.452131,0.431904,502.140,5.6000


## Annual

In [18]:
dates = dates.reset_index()
dates.columns = ['year', 'date']

df = pd.merge(dates, comps)
df = df[df['year']>1925]

In [19]:
df

,year,date,company,permno
20,1926,1925-12-31,Remington Typewriter,14525
21,1926,1925-12-31,American Car & Foundry,10006
22,1926,1925-12-31,American Locomotive,11287
23,1926,1925-12-31,Mack Truck,12941
24,1926,1925-12-31,U.S. Rubber,14912
...,...,...,...,...
2872,2021,2020-08-31,American Express,59176
2873,2021,2020-08-31,Merck & Co.,22752
2874,2021,2020-08-31,IBM,12490
2875,2021,2020-08-31,DuPont,16851


In [20]:
df.groupby('year')['year'].count().loc[lambda x: x!=30]

year
1926    20
1927    20
1929    29
1930    29
1931    29
Name: year, dtype: int64

In [21]:
dow = pd.merge(df, annretsdivs, how='left')

In [ ]:
dow[dow['ret'].isna()]

,year,date,company,permno,ret,retx,prc,div


In [23]:
annrets['f1ret'] = annrets.groupby('permno')['ret'].shift(-1)

**Note**: These returns don't include `dlret` due to mergers.

In [30]:
conn.raw_sql("""
    SELECT permno, date, dlpdt, nwperm, dlstcd, dlret
    FROM crsp.mse
    WHERE permno IN %(permnos)s
        AND event='DELIST'
        AND dlret>0
    """, params=params)

,permno,date,dlpdt,nwperm,dlstcd,dlret
0,10006.0,1984-06-28,1984-06-29,0.0,233.0,0.035629
1,10225.0,2014-04-30,2014-05-01,0.0,233.0,0.000359
2,10233.0,1986-01-03,1986-01-06,0.0,261.0,0.007412
3,10241.0,1988-12-15,1988-12-16,70519.0,241.0,0.004367
4,10364.0,1999-11-17,1999-11-18,0.0,233.0,0.008475
5,10401.0,2005-11-18,2005-11-21,66093.0,231.0,0.026009
6,10479.0,1984-02-08,1984-02-09,0.0,233.0,0.008043
7,10495.0,1977-01-12,1977-01-13,10604.0,241.0,0.125000
8,10989.0,2000-10-04,2000-10-05,0.0,233.0,0.003436
9,11260.0,1998-11-12,1998-11-13,86381.0,231.0,0.029990


In [144]:
dow = pd.merge(dow, annrets[['permno', 'year', 'f1ret']]).reindex(['company', 'year', 'prc', 'div', 'f1ret'], axis=1)

In [145]:
dow

,company,year,prc,div,f1ret
0,Remington Typewriter,1926,117.875,0.0000,-0.007679
1,American Car & Foundry,1926,101.500,8.3125,0.150657
2,American Locomotive,1926,109.500,8.0000,0.103556
3,Mack Truck,1926,97.750,12.1250,0.162948
4,U.S. Rubber,1926,59.000,0.0000,-0.038136
...,...,...,...,...,...
2849,American Express,2021,163.600,1.7200,-0.144965
2850,Merck & Co.,2021,76.640,6.3400,0.209941
2851,IBM,2021,133.660,11.8260,0.082436
2852,DuPont,2021,80.780,1.2000,-0.305715


In [147]:
dow.groupby('year')['company'].count().loc[lambda x: x!=30]

year
1926    20
1927    20
1929    29
1930    29
1931    29
1989    29
1990    29
2001    29
Name: company, dtype: int64

In [181]:
conn.raw_sql("""
    SELECT DISTINCT permno, namedt, nameendt, comnam
    FROM crsp.msenames
    WHERE permno = 10241
    ORDER BY namedt
    """)

,permno,namedt,nameendt,comnam
0,10241.0,1925-12-31,1962-07-01,AMERICAN CAN CO
1,10241.0,1962-07-02,1968-01-01,AMERICAN CAN CO
2,10241.0,1968-01-02,1987-04-27,AMERICAN CAN CO
3,10241.0,1987-04-28,1988-12-15,PRIMERICA CORP


In [187]:
conn.raw_sql("""
    SELECT date, nwperm, dlstcd, dlret, dlpdt
    FROM crsp.mse
    WHERE permno = 10241 AND event='DELIST'
    """).T

,0
date,1988-12-15
nwperm,70519.0
dlstcd,241.0
dlret,0.004367
dlpdt,1988-12-16


In [188]:
conn.raw_sql("""
    SELECT DISTINCT permno, namedt, nameendt, comnam
    FROM crsp.msenames
    WHERE permno = 70519
    ORDER BY namedt
    """)

,permno,namedt,nameendt,comnam
0,70519.0,1986-10-29,1988-05-05,COMMERCIAL CREDIT CO
1,70519.0,1988-05-06,1988-12-15,COMMERCIAL CREDIT GROUP INC
2,70519.0,1988-12-16,1989-01-19,PRIMERICA CORP NEW
3,70519.0,1989-01-20,1994-01-02,PRIMERICA CORP NEW
4,70519.0,1994-01-03,1995-04-26,TRAVELERS INC
5,70519.0,1995-04-27,1998-10-07,TRAVELERS GROUP INC
6,70519.0,1998-10-08,1998-12-03,CITIGROUP INC
7,70519.0,1998-12-04,2000-05-31,CITIGROUP INC
8,70519.0,2000-06-01,2002-01-01,CITIGROUP INC
9,70519.0,2002-01-02,2002-01-31,CITIGROUP INC


In [150]:
dow.to_excel('dow.xlsx')

# Foolish four

https://www.fool.com/investing/dividends-income/2006/02/14/fools-first-loves-the-foolish-four.aspx

"You start with the 30 stocks that make up the Dow Jones Industrial Average . This acts as a prescreen to ensure that you are selecting from successful, large-cap U.S. stocks. Then you screen those 30 stocks for the 10 with the highest dividend yield. (Yield equals dividend per share divided by price per share.) You then buy the five lowest-priced stocks of the high-yield 10 and hold them for one year. At that point, you recalculate the list, then make any necessary changes. And so forth. The original Foolish Four (a.k.a. Foolish 4.0) modified the Beating the Dow list by dropping the lowest-priced stock and doubling the investment in the second-lowest-priced stock."

In [136]:
dow['dy'] = dow['div'] / dow['prc']

In [124]:
dow['dy_rnk'] = dow.groupby('year')['dy'].rank(method='min', ascending=False)

In [126]:
dow[(dow['year']==2000) & (dow['dy_rnk']<10)].sort_values('prc')

,year,date,company,permno,ret,retx,prc,div,dy,dy_rnk
2219,2000,1999-11-01,AT&T,10401,-0.653439,-0.660517,17.2500,0.697500,0.040435,4.0
2201,2000,1999-11-01,Hewlett-Packard,27828,-0.304922,-0.309485,31.5625,29.951638,0.948963,1.0
2199,2000,1999-11-01,Alcoa,24643,-0.180157,-0.192771,33.5000,0.750000,0.022388,9.0
2221,2000,1999-11-01,Eastman Kodak,11754,-0.386215,-0.405660,39.3750,1.760000,0.044698,3.0
2208,2000,1999-11-01,International Paper,21573,-0.257134,-0.276855,40.8125,1.000000,0.024502,8.0
2200,2000,1999-11-01,Philip Morris Companies,13901,1.059337,0.913043,44.0000,2.020000,0.045909,2.0
2209,2000,1999-11-01,Caterpillar,18542,0.041532,0.005312,47.3125,1.330000,0.028111,7.0
2223,2000,1999-11-01,DuPont,11703,-0.245006,-0.266603,48.3125,1.400000,0.028978,6.0
2224,2000,1999-11-01,General Motors,12079,-0.278416,-0.299226,50.9375,2.000000,0.039264,5.0


In [116]:
dow['prc_rnk'] = dow.groupby('year')['prc'].rank(method='min', ascending=True)